# 5 · Search: BM25

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed").exists())
DATA = ROOT / "data" / "processed"
SEED = "Hundred Years' War"
BLUE, GRAY = "#2a78d6", "#b9b8b1"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12})

> ✏️ **[Your text]** What does the search bar do, and why BM25 rather than exact title matching?

Each article is a document: its **title plus first paragraph**. For a query $Q$ with words $q_i$, BM25 scores a document $D$:

$$\text{score}(D, Q) = \sum_i \text{IDF}(q_i) \cdot \frac{f(q_i, D)\,(k_1 + 1)}{f(q_i, D) + k_1 \left(1 - b + b\,\frac{|D|}{\text{avgdl}}\right)}, \qquad \text{IDF}(q) = \ln\frac{N}{\text{df}(q)}$$

with $k_1 = 1.2$ (how fast repeated words stop adding) and $b = 0.75$ (how much long documents are penalized).

In [2]:
import math, re

index = json.loads((ROOT / "docs" / "search.js").read_text().removeprefix("window.SEARCH = ").rstrip().rstrip(";"))
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
tokenize = lambda text: TOKEN_RE.findall(text.lower())

def bm25(query, k1=1.2, b=0.75):
    # The same formula as notebook 06 and the page.
    scores = np.zeros(index["N"])
    for word in set(tokenize(query)):
        if word not in index["postings"]:
            continue
        doc_ids, tfs = index["postings"][word]
        idf = math.log(index["N"] / len(doc_ids))
        for d, tf in zip(doc_ids, tfs):
            scores[d] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * index["length"][d] / index["avgdl"]))
    return scores

def top(query, k=5):
    s = bm25(query)
    best = np.argsort(-s)[:k]
    return pd.DataFrame({"article": [index["titles"][d] for d in best], "BM25": s[best].round(2),
                         "words": [index["length"][d] for d in best]}, index=range(1, k + 1))

top("Agincourt")

,article,BM25,words
1,Battle of Agincourt,6.27,90
2,Battle of Verneuil,5.35,57
3,Battle of La Brossinière,5.17,63
4,"John II, Duke of Alençon",4.67,82
5,Henry V of England,3.87,123


In [3]:
top("mad king")

,article,BM25,words
1,Charles VI of France,9.13,62
2,Henry II of Castile,2.81,70
3,John I of France,2.70,118
4,Charles IV of France,2.60,74
5,1383–1385 Portuguese interregnum,2.55,45


> ✏️ **[Your text]** Walk through one example: why is the top result first? What can BM25 not do (e.g. synonyms)?

---
**Full analysis, with all the code:** [06_search.ipynb](https://github.com/amos-johnson/hundred-years-war-similarity/blob/main/notebooks/06_search.ipynb)